# 53. RealMLP at the full internal ensemble

**One variable against ledger row 135** (`realmlp`, CV 0.967728): `N_ENS`, the number of
independently initialised models averaged within each fold, from **3 to 10**. Everything else is
held: same architecture, same 12 epochs, same batch 512, same learning rate, same schedule, same
EMA decay, same label smoothing, same preprocessing, same folds, same seed, same encoder
fingerprinted `0642e41750ef8bab`.

## Why this and not the whole public configuration

The public RealMLP runs 10 internal members across 3 seeds, which is 150 fits. Row 135 measured
this implementation at 2.0 minutes per fit, so the faithful version is five hours of compute at
our batch size and closer to eight at theirs. That is over the 9-hour guard once Kaggle's queue is
added, and notebook 43 already cost a wasted run to a projection that went wrong.

Cutting the seed loop and keeping the ensemble is the right trade. **The ensemble size is the
lever**: row 135's individual members score 0.9670 to 0.9679 per fold and their 3-member average
lands above every one of them, so the averaging is visibly still paying at 3. Multi-seed averaging
on top is second-order variance reduction, and rows 20 to 23 and row 32 already measured seed
averaging as near-null in this repo.

Keeping batch and epochs fixed also makes this genuinely one variable, which the faithful version
would not be.

## The prediction

**0.9683 to 0.9688.** Row 135 got 0.967728 at 3 members and the public figure is about 0.9688 at
10 members across 3 seeds, so 10 members at one seed should land most of the way and short of the
public number by the seed loop alone.

If it lands there it becomes **the best single model in this repo**, past `xgb_tuned` at 0.968222.

**In the stack I expect +0.00005 to +0.00015.** Row 136 showed `realmlp` entering at +0.2318, the
second largest coefficient, so a stronger version of that member should be worth something, but
the five-gate saturation curve says not to expect much: a better version of a member the stack
already holds is exactly what row 126 measured at twenty-one millionths.

**The case against**, which has beaten my predictions six times out of seven. Ensemble averaging
has diminishing returns by construction, and most of the variance reduction available from
averaging ten models is already captured by three. The gap between our 0.967728 and the public
0.9688 may be the seed loop, the batch size, or the 150-fit tuning behind their configuration
rather than the ensemble count, in which case this returns +0.0002 and not +0.0010.

In [ ]:
SMOKE = True

SEED = 42
N_SPLITS = 5
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

ARMS = ["realmlp"]

# Architecture. Public configuration where it is affordable, reduced where it is not.
HIDDEN = (768, 512, 512)
DROPOUT = 0.07
EMB_DIM = 8                 # categorical embedding width
PB_HIDDEN = 32              # periodic frequencies per numeric feature
PB_OUT = 6                  # output width per numeric feature after the periodic map
PB_FREQ_SCALE = 10.0
N_ENS = 10                  # THE ONE VARIABLE. Row 135 ran 3. Public runs 10 x 3 seeds.

# Optimisation.
EPOCHS, BATCH, LR, WD = 12, 512, 8e-3, 0.015
FLAT_RATIO = 0.3            # flat-cosine: flat for this fraction, then cosine to zero
EMA_DECAY = 0.997875
LS_EPS = 0.04               # label smoothing, cosine-annealed to zero
GRAD_CLIP = 1.2
SCALE_LR_MULT, BIAS_LR_MULT = 10.0, 0.1
SCALE_WD_MULT, BIAS_WD_MULT = 0.1, 0.5

ROW127_CV = 0.965798
ROW135_CV = 0.967728
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"

if SMOKE:
    EPOCHS, N_SPLITS, N_ENS = 2, 2, 2

print(f"SMOKE = {SMOKE}   epochs {EPOCHS}  n_ens {N_ENS}  hidden {HIDDEN}")
print("no validation is consulted at any point; the final EMA weights are used")

In [ ]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")

## The encoder, fingerprinted against 13

In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")

In [ ]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## The inputs, identical to row 127

In [ ]:
DST, SM_, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                    "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM_, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM_], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm

RB_TR = ratio_block(train).to_numpy(np.float32)
RB_TE = ratio_block(test).to_numpy(np.float32)
print(f"row 106 fed {len(NUM_COLS) + len(ENC_COLS)} numeric + {len(NUM_COLS)} mask columns")
print(f"these arms feed {len(NUM_COLS) + len(ENC_COLS) + len(RATIO_COLS)} numeric "
      f"+ {len(NUM_COLS)} mask")

cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)
Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)

## The architecture

Written from the paper's description. The pieces that differ from row 127 are the periodic
embedding of every numeric feature, the learnable front scale, and the parameter groups that give
scale and bias tensors their own learning rate and weight decay.

In [ ]:
import math


class PeriodicEmbedding(nn.Module):
    """Each numeric feature x_j becomes sin/cos of learnable frequencies, then its own
    linear map. This is the component row 127 has no equivalent of: it lets the network
    represent a non-monotone response to a single feature without spending depth on it."""

    def __init__(self, n_num, hidden=PB_HIDDEN, out=PB_OUT, freq_scale=PB_FREQ_SCALE):
        super().__init__()
        self.freq = nn.Parameter(torch.randn(n_num, hidden) * freq_scale)
        self.bias = nn.Parameter(torch.zeros(n_num, hidden))
        self.lin = nn.Parameter(torch.randn(n_num, 2 * hidden, out) / math.sqrt(2 * hidden))
        self.act = nn.PReLU(num_parameters=1)
        self.out_dim = n_num * out

    def forward(self, x):
        z = x.unsqueeze(-1) * self.freq + self.bias          # (B, n_num, hidden)
        e = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)  # (B, n_num, 2*hidden)
        o = torch.einsum("bnh,nho->bno", e, self.lin)        # (B, n_num, out)
        return self.act(o).flatten(1)


class FrontScale(nn.Module):
    """A learnable diagonal scale on the raw numeric block. Cheap, and it lets the network
    down-weight a useless column without having to cancel it inside the first linear."""

    def __init__(self, n):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n))

    def forward(self, x):
        return x * self.scale


class RealMLP(nn.Module):
    def __init__(self, n_num, cat_sizes, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.front = FrontScale(n_num)
        self.periodic = PeriodicEmbedding(n_num)
        self.embs = nn.ModuleList([nn.Embedding(s, EMB_DIM) for s in cat_sizes])
        dim = n_num + self.periodic.out_dim + EMB_DIM * len(cat_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        parts = [self.front(xn), self.periodic(xn)]
        parts += [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat(parts, dim=1)).squeeze(1)


def param_groups(model):
    """Scale and bias tensors get their own learning rate and decay, as the paper specifies."""
    scale, bias, rest = [], [], []
    for n, prm in model.named_parameters():
        if "scale" in n or isinstance(prm, nn.Parameter) and prm.ndim == 1 and "bias" not in n:
            scale.append(prm)
        elif n.endswith("bias"):
            bias.append(prm)
        else:
            rest.append(prm)
    return [
        {"params": rest, "lr": LR, "weight_decay": WD},
        {"params": scale, "lr": LR * SCALE_LR_MULT, "weight_decay": WD * SCALE_WD_MULT},
        {"params": bias, "lr": LR * BIAS_LR_MULT, "weight_decay": WD * BIAS_WD_MULT},
    ]


class EMA:
    """Evaluate a moving average of the weights rather than the last iterate."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v)


def flat_cos(step, total, flat=FLAT_RATIO):
    """Flat for the first `flat` fraction of training, then cosine to zero."""
    f = int(total * flat)
    if step < f:
        return 1.0
    return 0.5 * (1.0 + math.cos(math.pi * (step - f) / max(1, total - f)))


print("RealMLP defined. Components absent from row 127: periodic embeddings, front scale,")
print("parameter-group multipliers, EMA, flat-cosine schedule, label smoothing.")

## The preprocessing the architecture specifies

Median-centre, IQR-scale, smooth-clip. This replaces row 127's quantile transform. All three are
fit on training rows only, inside the fold. `smooth_clip` is a soft bound that keeps extreme
values finite without the hard cut a clip would apply, which matters because the periodic
embedding is sensitive to the scale of its input.

In [ ]:
def fit_prep(a):
    med = np.nanmedian(a, axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    q1, q3 = np.nanpercentile(a, [25, 75], axis=0)
    iqr = np.where(np.isnan(q3 - q1) | ((q3 - q1) < 1e-9), 1.0, q3 - q1)
    return med, iqr


def apply_prep(a, med, iqr, c=4.0):
    x = np.where(np.isnan(a), med, a)
    x = (x - med) / iqr
    return (x / np.sqrt(1.0 + (x / c) ** 2)).astype(np.float32)   # smooth clip


NUM_WORKERS = 0 if os.name == "nt" else 2


def make_loader(xn, xc, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(
        ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEV.type == "cuda"), drop_last=drop_last)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xn, xc, _ in loader:
        out.append(torch.sigmoid(model(xn.to(DEV), xc.to(DEV))).float().cpu().numpy())
    return np.concatenate(out)


LOG = OUT / "51_realmlp.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, device={DEV}, n_ens={N_ENS} ===")

In [ ]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))
per = []
t_start = time.time()

for f in range(N_SPLITS):
    tr_i = np.where(folds != f)[0]
    va_i = np.where(folds == f)[0]
    Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
    num_tr = np.hstack([Etr[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[tr_i]])
    num_va = np.hstack([Eva[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[va_i]])
    num_te = np.hstack([Ete[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TE])

    med, iqr = fit_prep(num_tr)
    Xn_tr = np.hstack([apply_prep(num_tr, med, iqr), mask_tr[tr_i]])
    Xn_va = np.hstack([apply_prep(num_va, med, iqr), mask_tr[va_i]])
    Xn_te = np.hstack([apply_prep(num_te, med, iqr), mask_te])

    tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], y[tr_i], BATCH, True, drop_last=True)
    va_loader = make_loader(Xn_va, Xc_tr[va_i], None, BATCH * 8, False)
    te_loader = make_loader(Xn_te, Xc_te, None, BATCH * 8, False)

    va_ens = np.zeros(len(va_i))
    te_ens = np.zeros(len(test))
    for e in range(N_ENS):
        seed_all(SEED + 100 * f + e)
        model = RealMLP(Xn_tr.shape[1], cat_sizes).to(DEV)
        opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.98))
        total = EPOCHS * len(tr_loader)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: flat_cos(s, total))
        ema = EMA(model)
        step = 0
        for ep in range(EPOCHS):
            model.train()
            eps = LS_EPS * 0.5 * (1 + math.cos(math.pi * ep / max(1, EPOCHS - 1)))
            for xn, xc, yy in tr_loader:
                yy = yy.to(DEV)
                yy = yy * (1 - eps) + 0.5 * eps
                opt.zero_grad(set_to_none=True)
                loss = nn.functional.binary_cross_entropy_with_logits(
                    model(xn.to(DEV), xc.to(DEV)), yy)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
                sched.step()
                ema.update(model)
                step += 1
        ema.copy_to(model)
        pv = predict(model, va_loader)
        va_ens += pv / N_ENS
        te_ens += predict(model, te_loader) / N_ENS
        print(f"  fold {f} ens {e}: AUC {roc_auc_score(y[va_i], pv):.6f}")
        del model, ema
        if DEV.type == "cuda":
            torch.cuda.empty_cache()

    oof[va_i] = va_ens
    test_pred += te_ens / N_SPLITS
    per.append(float(roc_auc_score(y[va_i], va_ens)))
    note(f"fold {f}: ensembled AUC {per[-1]:.6f}   elapsed {(time.time()-t_start)/60:.1f} min")

per = np.array(per)
note(f"realmlp10: CV {per.mean():.6f} +/- {per.std():.6f} in {(time.time()-t_start)/60:.1f} min")

In [ ]:
def load_saved(stem):
    for name in (f"{stem}_oof.npy", f"{stem}.npy"):
        try:
            return np.load(locate(name))
        except FileNotFoundError:
            continue
    return None


print(f"realmlp CV {per.mean():.6f} +/- {per.std():.6f}")
print(f"row 135 realmlp (3 members) {ROW135_CV:.6f}, difference {per.mean() - ROW135_CV:+.6f}")

base = load_saved("realmlp")
if base is not None and not SMOKE:
    bp = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(N_SPLITS)])
    d = per - bp
    sd = d.std(ddof=1)
    print(f"row 135 re-scored: {bp.mean():.6f}, delta {bp.mean() - ROW135_CV:+.2e}")
    print(f"paired {d.mean():+.6f}, sd {sd:.6f}, {int((d > 0).sum())}/{N_SPLITS} folds"
          + (f", t={d.mean()/(sd/np.sqrt(N_SPLITS)):.2f}" if sd > 0 else ""))

print("\nwhere this sits among our models:")
for n, v in [("cat_te_fe", 0.968036), ("xgb_tuned", 0.968222), ("neural_fe", 0.965798),
             ("neural_te", 0.965373), ("neural (row 16)", 0.939169)]:
    print(f"  {n:18}{v:.6f}")
print(f"  {'realmlp (this)':18}{per.mean():.6f}")
print(f"\n  public single RealMLP on this split: about 0.9688 out-of-fold, LB 0.97009")
print(f"  our 51-member stack:                 0.968850 out-of-fold, LB 0.97014")

print("\ndisagreement with the members it would join:")
for b in ["neural_fe", "neural_lookup", "xgb_tuned", "cat_te_fe", "cat_native_c2"]:
    v = load_saved(b)
    if v is None:
        continue
    v = v[ROW_IDX] if len(v) != len(oof) else v
    print(f"  {b:16}{pd.Series(oof).corr(pd.Series(v), method='spearman'):.6f}")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
np.save(OUT / f"{pre}realmlp10_oof.npy", oof)
np.save(OUT / f"{pre}realmlp10_test.npy", test_pred)
print(f"wrote {pre}realmlp10_oof.npy, {pre}realmlp10_test.npy")
print(f"\nledger lines:\n  name    realmlp\n  cv_mean {per.mean():.6f}\n  cv_std  {per.std():.6f}")
print(f"\n  encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"ratio block pure {BLOCK_OK}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is a separate notebook and a separate ledger row.")